In [1]:
import requests

import logging
import json

In [ ]:
import requests
import os
from dotenv import load_dotenv
from modules.get_token import get_token
load_dotenv()
start_url = os.getenv("PRJ_START_URL")

In [ ]:
#======= test customer Purchase Order module

def test_simple_cpo_query(token):
    """Test simplest customer purchase order query"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    # Try different possible query names
    query_names = [
        "customerPurchaseOrders",
        "customerOrders", 
        "purchaseOrders",
        "orders",
        "salesOrders",
        "cpos"
    ]
    
    print("\n" + "="*60)
    print("TESTING CUSTOMER PURCHASE ORDER QUERY NAMES")
    print("="*60)
    
    valid_query_name = None
    
    for query_name in query_names:
        simple_query = f"""
        query {{
            {query_name} {{
                totalRecords
            }}
        }}
        """
        
        payload = {
            "query": simple_query
        }
        
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
            res.raise_for_status()
            data = res.json()
            
            if 'errors' in data:
                print(f"  ❌ {query_name}: Not available")
            elif 'data' in data and query_name in data['data']:
                total = data['data'][query_name].get('totalRecords', 'N/A')
                print(f"  ✓ {query_name}: VALID (Total Records: {total})")
                valid_query_name = query_name
                break
        
        except Exception as e:
            print(f"  ❌ {query_name}: Error - {e}")
    
    return valid_query_name


def test_cpo_fields(token, query_name):
    """Test common customer purchase order fields"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    # Common CPO field names to test
    test_fields = [
        "id",
        "number",
        "poNumber",
        "orderNumber",
        "customerPONumber",
        "customerOrderNumber",
        "customer",
        "customerName",
        "customerNumber",
        "clientPlainText",
        "clientPartNumber",
        "status",
        "statusLabel",
        "orderDate",
        "dateOrdered",
        "dueDate",
        "dateDue",
        "shipDate",
        "dateShipped",
        "promiseDate",
        "quantity",
        "quantityOrdered",
        "qtyOrdered",
        "quantityShipped",
        "qtyShipped",
        "partNumber",
        "partName",
        "part",
        "price",
        "unitPrice",
        "totalPrice",
        "totalAmount",
        "notes",
        "description",
        "revision",
        "rev",
        "salesPerson",
        "salesperson",
        "contact",
        "priority",
        "urgent"
    ]
    
    valid_fields = []
    
    print("\n" + "="*60)
    print(f"TESTING {query_name.upper()} FIELDS")
    print("="*60)
    
    for field in test_fields:
        query = f"""
        query {{
            {query_name}(pageSize: 1) {{
                records {{
                    {field}
                }}
                totalRecords
            }}
        }}
        """
        
        payload = {
            "query": query
        }
        
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
            res.raise_for_status()
            data = res.json()
            
            if 'errors' in data:
                print(f"  ❌ {field}: Not available")
            elif 'data' in data:
                print(f"  ✓ {field}: VALID")
                valid_fields.append(field)
                # Show sample value
                if data['data'][query_name]['records']:
                    sample_value = data['data'][query_name]['records'][0].get(field)
                    if sample_value:
                        sample_str = str(sample_value)[:50]
                        print(f"      Sample: {sample_str}")
        
        except Exception as e:
            print(f"  ❌ {field}: Error")
    
    return valid_fields


def get_cpo_with_valid_fields(token, query_name, fields, page_size=10):
    """Get customer purchase orders using valid fields"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    fields_str = "\n                ".join(fields)
    
    query = f"""
    query {query_name}($pageSize: Int, $pageStart: Int) {{
        {query_name}(pageSize: $pageSize, pageStart: $pageStart) {{
            records {{
                {fields_str}
            }}
            pageSize
            pageStart
            totalRecords
        }}
    }}
    """
    
    payload = {
        "query": query,
        "variables": {
            "pageSize": page_size,
            "pageStart": 0
        }
    }
    
    try:
        res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
        res.raise_for_status()
        
        data = res.json()
        
        if 'errors' in data:
            logging.error(f"GraphQL Errors: {data['errors']}")
            return None
        elif 'data' in data:
            cpo_data = data['data'][query_name]
            logging.info(f"✓ SUCCESS! Retrieved {len(cpo_data['records'])} orders")
            logging.info(f"✓ Total Records: {cpo_data['totalRecords']}")
            
            print("\n" + "="*60)
            print("SAMPLE CUSTOMER PURCHASE ORDERS:")
            print("="*60)
            for i, order in enumerate(cpo_data['records'][:5], 1):
                print(f"\nOrder {i}:")
                print(json.dumps(order, indent=2))
            
            # Save the working query
            with open('working_cpo_query.txt', 'w') as f:
                f.write(query)
            print("\n✓ Working query saved to 'working_cpo_query.txt'")
            
            # Save sample data
            with open('cpo_sample_data.json', 'w') as f:
                json.dump(cpo_data['records'][:10], f, indent=2)
            print("✓ Sample data saved to 'cpo_sample_data.json'")
            
            return data
        
    except Exception as e:
        logging.error(f"Request failed: {e}")
        return None


def test_cpo_with_filters(token, query_name, fields):
    """Test customer purchase orders with filters"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    fields_str = "\n                ".join(fields[:5])  # Use first 5 valid fields
    
    # Test with filter parameter
    query_with_filter = f"""
    query {query_name}($filter: CustomerPurchaseOrderFilter, $pageSize: Int) {{
        {query_name}(filter: $filter, pageSize: $pageSize) {{
            records {{
                {fields_str}
            }}
            totalRecords
        }}
    }}
    """
    
    payload = {
        "query": query_with_filter,
        "variables": {
            "filter": {},
            "pageSize": 5
        }
    }
    
    try:
        res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
        res.raise_for_status()
        
        data = res.json()
        
        if 'errors' in data:
            print("\n❌ Filter parameter not supported or incorrect type")
            print(f"Error: {data['errors']}")
            return False
        else:
            print("\n✓ Filter parameter is supported!")
            return True
        
    except Exception as e:
        print(f"\n❌ Filter test error: {e}")
        return False


def get_all_cpo_paginated(token, query_name, fields, page_size=100):
    """Fetch all customer purchase orders with pagination"""
    all_orders = []
    page_start = 0

    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    fields_str = "\n                ".join(fields)
    
    query = f"""
    query {query_name}($pageSize: Int, $pageStart: Int) {{
        {query_name}(pageSize: $pageSize, pageStart: $pageStart) {{
            records {{
                {fields_str}
            }}
            pageSize
            pageStart
            totalRecords
        }}
    }}
    """
    
    while True:
        payload = {
            "query": query,
            "variables": {
                "pageSize": page_size,
                "pageStart": page_start
            }
        }
        
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
            res.raise_for_status()
            
            data = res.json()
            
            if 'data' in data and query_name in data['data']:
                order_data = data['data'][query_name]
                records = order_data['records']
                total_records = order_data['totalRecords']
                
                all_orders.extend(records)
                
                logging.info(f"Fetched {len(records)} records (Total: {len(all_orders)}/{total_records})")
                
                # Check if we've fetched all records
                if page_start + page_size >= total_records or not records:
                    break
                
                page_start += page_size
            else:
                break
        
        except Exception as e:
            logging.error(f"Error during pagination: {e}")
            break
    
    logging.info(f"Total orders fetched: {len(all_orders)}")
    return all_orders


if __name__ == "__main__":
    token = get_token()
    if token:
        print("\n🔍 PROSHOP CUSTOMER PURCHASE ORDER MODULE - COMPLETE TESTING")
        print("="*60)
        
        # Step 1: Find the correct query name
        print("\n[STEP 1] Finding correct query name...")
        query_name = test_simple_cpo_query(token)
        
        if query_name:
            print(f"\n✓ Found query name: {query_name}")
            
            # Step 2: Test all possible fields
            print(f"\n[STEP 2] Testing fields for {query_name}...")
            valid_fields = test_cpo_fields(token, query_name)
            
            print("\n" + "="*60)
            print(f"ALL VALID FIELDS ({len(valid_fields)}):")
            print("="*60)
            for field in valid_fields:
                print(f"  • {field}")
            
            if valid_fields:
                # Step 3: Get sample data
                print(f"\n[STEP 3] Retrieving sample {query_name}...")
                get_cpo_with_valid_fields(token, query_name, valid_fields, page_size=10)
                
                # Step 4: Test filters
                print(f"\n[STEP 4] Testing filter support...")
                test_cpo_with_filters(token, query_name, valid_fields)
                
                # Step 5: Option to fetch all
                print("\n" + "="*60)
                print("READY TO FETCH ALL ORDERS")
                print("="*60)
                print("\nTo fetch all orders, uncomment the following:")
                print(f"# all_orders = get_all_cpo_paginated(token, '{query_name}', valid_fields, page_size=100)")
                
                # Uncomment to fetch all:
                # all_orders = get_all_cpo_paginated(token, query_name, valid_fields, page_size=100)
                # print(f"\n✓ Fetched all {len(all_orders)} orders!")
                
        else:
            print("\n❌ Could not find customer purchase order query.")
            print("Try checking the Proshop UI or documentation for the correct module name.")


🔍 PROSHOP CUSTOMER PURCHASE ORDER MODULE - COMPLETE TESTING

[STEP 1] Finding correct query name...

TESTING CUSTOMER PURCHASE ORDER QUERY NAMES
  ❌ customerPurchaseOrders: Not available
  ❌ customerOrders: Not available
  ✓ purchaseOrders: VALID (Total Records: 4893)

✓ Found query name: purchaseOrders

[STEP 2] Testing fields for purchaseOrders...

TESTING PURCHASEORDERS FIELDS
  ✓ id: VALID
      Sample: 161869
  ❌ number: Not available
  ❌ poNumber: Not available
  ❌ orderNumber: Not available
  ❌ customerPONumber: Not available
  ❌ customerOrderNumber: Not available
  ❌ customer: Not available
  ❌ customerName: Not available
  ❌ customerNumber: Not available
  ❌ clientPlainText: Not available
  ❌ clientPartNumber: Not available
  ❌ status: Not available
  ❌ statusLabel: Not available
  ❌ orderDate: Not available
  ❌ dateOrdered: Not available
  ❌ dueDate: Not available
  ❌ dateDue: Not available
  ❌ shipDate: Not available
  ❌ dateShipped: Not available
  ❌ promiseDate: Not avail

In [ ]:
#======= test equipment module
def test_simple_equipment_query(token):
    """Test simplest equipment query"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    # Try different possible query names
    query_names = [
        "equipment",
        "equipments",
        "machines",
        "machinery",
        "assets",
        "resources",
        "tools",
        "workCenters",
        "workcenters"
    ]
    
    print("\n" + "="*60)
    print("TESTING EQUIPMENT QUERY NAMES")
    print("="*60)
    
    valid_query_name = None
    
    for query_name in query_names:
        simple_query = f"""
        query {{
            {query_name} {{
                totalRecords
            }}
        }}
        """
        
        payload = {
            "query": simple_query
        }
        
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
            res.raise_for_status()
            data = res.json()
            
            if 'errors' in data:
                print(f"  ❌ {query_name}: Not available")
            elif 'data' in data and query_name in data['data']:
                total = data['data'][query_name].get('totalRecords', 'N/A')
                print(f"  ✓ {query_name}: VALID (Total Records: {total})")
                valid_query_name = query_name
                break
        
        except Exception as e:
            print(f"  ❌ {query_name}: Error - {e}")
    
    return valid_query_name


def test_equipment_fields(token, query_name):
    """Test common equipment fields"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    # Common equipment field names to test
    test_fields = [
        "id",
        "number",
        "equipmentNumber",
        "internalPartNumber",
        "assetNumber",
        "name",
        "equipmentName",
        "description",
        "type",
        "equipmentType",
        "category",
        "status",
        "statusLabel",
        "active",
        "isActive",
        "manufacturer",
        "model",
        "modelNumber",
        "serialNumber",
        "serial",
        "location",
        "department",
        "workCenter",
        "workcenter",
        "cost",
        "purchaseCost",
        "value",
        "purchaseDate",
        "datePurchased",
        "installDate",
        "dateInstalled",
        "warrantyExpiration",
        "warrantyDate",
        "lastMaintenance",
        "lastMaintenanceDate",
        "nextMaintenance",
        "nextMaintenanceDate",
        "maintenanceSchedule",
        "hoursUsed",
        "totalHours",
        "cyclesCompleted",
        "utilization",
        "notes",
        "specifications",
        "capacity",
        "owner",
        "operator",
        "assignedTo",
        "vendor",
        "supplier"
    ]
    
    valid_fields = []
    
    print("\n" + "="*60)
    print(f"TESTING {query_name.upper()} FIELDS")
    print("="*60)
    
    for field in test_fields:
        query = f"""
        query {{
            {query_name}(pageSize: 1) {{
                records {{
                    {field}
                }}
                totalRecords
            }}
        }}
        """
        
        payload = {
            "query": query
        }
        
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
            res.raise_for_status()
            data = res.json()
            
            if 'errors' in data:
                print(f"  ❌ {field}: Not available")
            elif 'data' in data:
                print(f"  ✓ {field}: VALID")
                valid_fields.append(field)
                # Show sample value
                if data['data'][query_name]['records']:
                    sample_value = data['data'][query_name]['records'][0].get(field)
                    if sample_value:
                        sample_str = str(sample_value)[:50]
                        print(f"      Sample: {sample_str}")
        
        except Exception as e:
            print(f"  ❌ {field}: Error")
    
    return valid_fields


def get_equipment_with_valid_fields(token, query_name, fields, page_size=10):
    """Get equipment using valid fields"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    fields_str = "\n                ".join(fields)
    
    query = f"""
    query {query_name}($pageSize: Int, $pageStart: Int) {{
        {query_name}(pageSize: $pageSize, pageStart: $pageStart) {{
            records {{
                {fields_str}
            }}
            pageSize
            pageStart
            totalRecords
        }}
    }}
    """
    
    payload = {
        "query": query,
        "variables": {
            "pageSize": page_size,
            "pageStart": 0
        }
    }
    
    try:
        res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
        res.raise_for_status()
        
        data = res.json()
        
        if 'errors' in data:
            logging.error(f"GraphQL Errors: {data['errors']}")
            return None
        elif 'data' in data:
            equipment_data = data['data'][query_name]
            logging.info(f"✓ SUCCESS! Retrieved {len(equipment_data['records'])} equipment records")
            logging.info(f"✓ Total Records: {equipment_data['totalRecords']}")
            
            print("\n" + "="*60)
            print("SAMPLE EQUIPMENT DATA:")
            print("="*60)
            for i, equipment in enumerate(equipment_data['records'][:5], 1):
                print(f"\nEquipment {i}:")
                print(json.dumps(equipment, indent=2))
            
            # Save the working query
            with open('working_equipment_query.txt', 'w') as f:
                f.write(query)
            print("\n✓ Working query saved to 'working_equipment_query.txt'")
            
            # Save sample data
            with open('equipment_sample_data.json', 'w') as f:
                json.dump(equipment_data['records'][:10], f, indent=2)
            print("✓ Sample data saved to 'equipment_sample_data.json'")
            
            return data
        
    except Exception as e:
        logging.error(f"Request failed: {e}")
        return None


def test_equipment_with_filters(token, query_name, fields):
    """Test equipment with filters"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    fields_str = "\n                ".join(fields[:5])  # Use first 5 valid fields
    
    # Test with filter parameter
    query_with_filter = f"""
    query {query_name}($filter: EquipmentFilter, $pageSize: Int) {{
        {query_name}(filter: $filter, pageSize: $pageSize) {{
            records {{
                {fields_str}
            }}
            totalRecords
        }}
    }}
    """
    
    payload = {
        "query": query_with_filter,
        "variables": {
            "filter": {},
            "pageSize": 5
        }
    }
    
    try:
        res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
        res.raise_for_status()
        
        data = res.json()
        
        if 'errors' in data:
            print("\n❌ Filter parameter not supported or incorrect type")
            print(f"Error: {data['errors']}")
            return False
        else:
            print("\n✓ Filter parameter is supported!")
            return True
        
    except Exception as e:
        print(f"\n❌ Filter test error: {e}")
        return False


def test_equipment_query_parameter(token, query_name, fields):
    """Test equipment with query parameter"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    fields_str = "\n                ".join(fields[:5])
    
    # Test with query parameter
    query_with_param = f"""
    query {query_name}($query: EquipmentQuery, $pageSize: Int) {{
        {query_name}(query: $query, pageSize: $pageSize) {{
            records {{
                {fields_str}
            }}
            totalRecords
        }}
    }}
    """
    
    payload = {
        "query": query_with_param,
        "variables": {
            "query": {},
            "pageSize": 5
        }
    }
    
    try:
        res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
        res.raise_for_status()
        
        data = res.json()
        
        if 'errors' in data:
            print("❌ Query parameter not supported or incorrect type")
            return False
        else:
            print("✓ Query parameter is supported!")
            return True
        
    except Exception as e:
        print(f"❌ Query test error: {e}")
        return False


def get_all_equipment_paginated(token, query_name, fields, page_size=100):
    """Fetch all equipment with pagination"""
    all_equipment = []
    page_start = 0

    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    fields_str = "\n                ".join(fields)
    
    query = f"""
    query {query_name}($pageSize: Int, $pageStart: Int) {{
        {query_name}(pageSize: $pageSize, pageStart: $pageStart) {{
            records {{
                {fields_str}
            }}
            pageSize
            pageStart
            totalRecords
        }}
    }}
    """
    
    while True:
        payload = {
            "query": query,
            "variables": {
                "pageSize": page_size,
                "pageStart": page_start
            }
        }
        
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
            res.raise_for_status()
            
            data = res.json()
            
            if 'data' in data and query_name in data['data']:
                equipment_data = data['data'][query_name]
                records = equipment_data['records']
                total_records = equipment_data['totalRecords']
                
                all_equipment.extend(records)
                
                logging.info(f"Fetched {len(records)} records (Total: {len(all_equipment)}/{total_records})")
                
                # Check if we've fetched all records
                if page_start + page_size >= total_records or not records:
                    break
                
                page_start += page_size
            else:
                break
        
        except Exception as e:
            logging.error(f"Error during pagination: {e}")
            break
    
    logging.info(f"Total equipment fetched: {len(all_equipment)}")
    return all_equipment


if __name__ == "__main__":
    token = get_token()
    if token:
        print("\n🔧 PROSHOP EQUIPMENT MODULE - COMPLETE TESTING")
        print("="*60)
        
        # Step 1: Find the correct query name
        print("\n[STEP 1] Finding correct query name...")
        query_name = test_simple_equipment_query(token)
        
        if query_name:
            print(f"\n✓ Found query name: {query_name}")
            
            # Step 2: Test all possible fields
            print(f"\n[STEP 2] Testing fields for {query_name}...")
            valid_fields = test_equipment_fields(token, query_name)
            
            print("\n" + "="*60)
            print(f"ALL VALID FIELDS ({len(valid_fields)}):")
            print("="*60)
            for field in valid_fields:
                print(f"  • {field}")
            
            if valid_fields:
                # Step 3: Get sample data
                print(f"\n[STEP 3] Retrieving sample {query_name}...")
                get_equipment_with_valid_fields(token, query_name, valid_fields, page_size=10)
                
                # Step 4: Test filters
                print(f"\n[STEP 4] Testing filter support...")
                test_equipment_with_filters(token, query_name, valid_fields)
                
                # Step 5: Test query parameter
                print(f"\n[STEP 5] Testing query parameter support...")
                test_equipment_query_parameter(token, query_name, valid_fields)
                
                # Step 6: Option to fetch all
                print("\n" + "="*60)
                print("READY TO FETCH ALL EQUIPMENT")
                print("="*60)
                print("\nTo fetch all equipment, uncomment the following:")
                print(f"# all_equipment = get_all_equipment_paginated(token, '{query_name}', valid_fields, page_size=100)")
                
                # Uncomment to fetch all:
                # all_equipment = get_all_equipment_paginated(token, query_name, valid_fields, page_size=100)
                # print(f"\n✓ Fetched all {len(all_equipment)} equipment records!")
                # with open('all_equipment.json', 'w') as f:
                #     json.dump(all_equipment, f, indent=2)
                # print("✓ All equipment data saved to 'all_equipment.json'")
                
        else:
            print("\n❌ Could not find equipment query.")
            print("Try checking the Proshop UI or documentation for the correct module name.")
            print("\nPossible reasons:")
            print("  • Equipment might be under a different module name")
            print("  • Your account might not have access to equipment data")
            print("  • Equipment might be called 'assets', 'resources', or 'workCenters'")

2026-01-12 22:49:28,797 - INFO - Deriving session token
2026-01-12 22:49:31,240 - INFO - succesfully derived session token



🔧 PROSHOP EQUIPMENT MODULE - COMPLETE TESTING

[STEP 1] Finding correct query name...

TESTING EQUIPMENT QUERY NAMES
  ❌ equipment: Not available
  ✓ equipments: VALID (Total Records: 727)

✓ Found query name: equipments

[STEP 2] Testing fields for equipments...

TESTING EQUIPMENTS FIELDS
  ❌ id: Not available
  ❌ number: Not available
  ✓ equipmentNumber: VALID
      Sample: 250100
  ❌ internalPartNumber: Not available
  ❌ assetNumber: Not available
  ❌ name: Not available
  ❌ equipmentName: Not available
  ✓ description: VALID
  ✓ type: VALID
      Sample: Thread Sealer
  ✓ equipmentType: VALID
      Sample: SLP
  ❌ category: Not available
  ✓ status: VALID
      Sample: Active
  ❌ statusLabel: Not available
  ❌ active: Not available
  ❌ isActive: Not available
  ❌ manufacturer: Not available
  ❌ model: Not available
  ❌ modelNumber: Not available
  ✓ serialNumber: VALID
  ❌ serial: Not available
  ✓ location: VALID
      Sample: SC5
  ❌ department: Not available
  ❌ workCenter: No

2026-01-12 22:50:47,043 - INFO - ✓ SUCCESS! Retrieved 10 equipment records
2026-01-12 22:50:47,044 - INFO - ✓ Total Records: 727



SAMPLE EQUIPMENT DATA:

Equipment 1:
{
  "description": "",
  "equipmentNumber": "250100",
  "equipmentType": "SLP",
  "location": "SC5",
  "notes": "",
  "serialNumber": "",
  "status": "Active",
  "type": "Thread Sealer"
}

Equipment 2:
{
  "description": "DAP\u00ae Joint Sealant: 2.8 oz Tube, Clear, RTV Silicone",
  "equipmentNumber": "2572345",
  "equipmentType": "SLP",
  "location": "SC5",
  "notes": "",
  "serialNumber": "",
  "status": "Active",
  "type": "DAP\u00ae Joint Sealant: 2.8 oz Tube, Clear, RTV Silicone"
}

Equipment 3:
{
  "description": "",
  "equipmentNumber": "21884",
  "equipmentType": "QA",
  "location": "",
  "notes": "<p>Reference Calibration WI-2 for calibration standards and procedures</p>\n\n<p>Record temperature and humidity</p>",
  "serialNumber": "",
  "status": "Active",
  "type": ""
}

Equipment 4:
{
  "description": "",
  "equipmentNumber": "1329837",
  "equipmentType": "SLP",
  "location": "SC5",
  "notes": "",
  "serialNumber": "",
  "status": "Acti

In [ ]:
#======= test COTS module
def test_simple_cots_query(token):
    """Test simplest COTS query"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    # Try different possible query names
    query_names = [
        "cots",
        "cotsItems",
        "cotsparts",
        "cotsParts",
        "commercialParts",
        "offTheShelf",
        "purchasedParts",
        "buyParts",
        "vendorParts",
        "suppliers",
        "supplierParts"
    ]
    
    print("\n" + "="*60)
    print("TESTING COTS QUERY NAMES")
    print("="*60)
    
    valid_query_name = None
    
    for query_name in query_names:
        simple_query = f"""
        query {{
            {query_name} {{
                totalRecords
            }}
        }}
        """
        
        payload = {
            "query": simple_query
        }
        
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
            res.raise_for_status()
            data = res.json()
            
            if 'errors' in data:
                print(f"  ❌ {query_name}: Not available")
            elif 'data' in data and query_name in data['data']:
                total = data['data'][query_name].get('totalRecords', 'N/A')
                print(f"  ✓ {query_name}: VALID (Total Records: {total})")
                valid_query_name = query_name
                break
        
        except Exception as e:
            print(f"  ❌ {query_name}: Error - {e}")
    
    return valid_query_name


def test_cots_fields(token, query_name):
    """Test common COTS fields"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    # Common COTS field names to test
    test_fields = [
        "id",
        "number",
        "partNumber",
        "cotsNumber",
        "itemNumber",
        "name",
        "cotsName",
        "itemName",
        "description",
        "revision",
        "rev",
        "status",
        "statusLabel",
        "active",
        "isActive",
        "manufacturer",
        "manufacturerName",
        "manufacturerPartNumber",
        "mfgPartNumber",
        "supplier",
        "supplierName",
        "vendor",
        "vendorName",
        "vendorPartNumber",
        "supplierPartNumber",
        "cost",
        "unitCost",
        "price",
        "unitPrice",
        "lastCost",
        "standardCost",
        "leadTime",
        "leadTimeDays",
        "minimumOrderQty",
        "minOrderQty",
        "orderMultiple",
        "unitOfMeasure",
        "uom",
        "onHandQty",
        "quantityOnHand",
        "availableQty",
        "allocatedQty",
        "onOrderQty",
        "reorderPoint",
        "reorderQty",
        "safetyStock",
        "location",
        "binLocation",
        "warehouse",
        "category",
        "commodityCode",
        "notes",
        "specifications",
        "weight",
        "dimensions",
        "dateCreated",
        "dateModified",
        "lastPurchaseDate",
        "lastPurchasePrice",
        "preferredVendor",
        "alternateVendors"
    ]
    
    valid_fields = []
    
    print("\n" + "="*60)
    print(f"TESTING {query_name.upper()} FIELDS")
    print("="*60)
    
    for field in test_fields:
        query = f"""
        query {{
            {query_name}(pageSize: 1) {{
                records {{
                    {field}
                }}
                totalRecords
            }}
        }}
        """
        
        payload = {
            "query": query
        }
        
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
            res.raise_for_status()
            data = res.json()
            
            if 'errors' in data:
                print(f"  ❌ {field}: Not available")
            elif 'data' in data:
                print(f"  ✓ {field}: VALID")
                valid_fields.append(field)
                # Show sample value
                if data['data'][query_name]['records']:
                    sample_value = data['data'][query_name]['records'][0].get(field)
                    if sample_value:
                        sample_str = str(sample_value)[:50]
                        print(f"      Sample: {sample_str}")
        
        except Exception as e:
            print(f"  ❌ {field}: Error")
    
    return valid_fields


def get_cots_with_valid_fields(token, query_name, fields, page_size=10):
    """Get COTS items using valid fields"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    fields_str = "\n                ".join(fields)
    
    query = f"""
    query {query_name}($pageSize: Int, $pageStart: Int) {{
        {query_name}(pageSize: $pageSize, pageStart: $pageStart) {{
            records {{
                {fields_str}
            }}
            pageSize
            pageStart
            totalRecords
        }}
    }}
    """
    
    payload = {
        "query": query,
        "variables": {
            "pageSize": page_size,
            "pageStart": 0
        }
    }
    
    try:
        res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
        res.raise_for_status()
        
        data = res.json()
        
        if 'errors' in data:
            logging.error(f"GraphQL Errors: {data['errors']}")
            return None
        elif 'data' in data:
            cots_data = data['data'][query_name]
            logging.info(f"✓ SUCCESS! Retrieved {len(cots_data['records'])} COTS items")
            logging.info(f"✓ Total Records: {cots_data['totalRecords']}")
            
            print("\n" + "="*60)
            print("SAMPLE COTS DATA:")
            print("="*60)
            for i, item in enumerate(cots_data['records'][:5], 1):
                print(f"\nCOTS Item {i}:")
                print(json.dumps(item, indent=2))
            
            # Save the working query
            with open('working_cots_query.txt', 'w') as f:
                f.write(query)
            print("\n✓ Working query saved to 'working_cots_query.txt'")
            
            # Save sample data
            with open('cots_sample_data.json', 'w') as f:
                json.dump(cots_data['records'][:10], f, indent=2)
            print("✓ Sample data saved to 'cots_sample_data.json'")
            
            return data
        
    except Exception as e:
        logging.error(f"Request failed: {e}")
        return None


def test_cots_with_filters(token, query_name, fields):
    """Test COTS with filters"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    fields_str = "\n                ".join(fields[:5])  # Use first 5 valid fields
    
    # Test with filter parameter
    query_with_filter = f"""
    query {query_name}($filter: COTSFilter, $pageSize: Int) {{
        {query_name}(filter: $filter, pageSize: $pageSize) {{
            records {{
                {fields_str}
            }}
            totalRecords
        }}
    }}
    """
    
    payload = {
        "query": query_with_filter,
        "variables": {
            "filter": {},
            "pageSize": 5
        }
    }
    
    try:
        res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
        res.raise_for_status()
        
        data = res.json()
        
        if 'errors' in data:
            print("\n❌ COTSFilter parameter not supported or incorrect type")
            print(f"Error: {data['errors']}")
            
            # Try alternative filter names
            alternative_filters = ["CotsFilter", "CotFilter", "CotsItemFilter"]
            for alt_filter in alternative_filters:
                print(f"\nTrying alternative: {alt_filter}")
                alt_query = query_with_filter.replace("COTSFilter", alt_filter)
                alt_payload = {"query": alt_query, "variables": {"filter": {}, "pageSize": 5}}
                
                try:
                    alt_res = requests.post(url, headers=headers, json=alt_payload, timeout=(10, 30))
                    alt_res.raise_for_status()
                    alt_data = alt_res.json()
                    
                    if 'errors' not in alt_data:
                        print(f"✓ {alt_filter} works!")
                        return True
                except:
                    continue
            
            return False
        else:
            print("\n✓ COTSFilter parameter is supported!")
            return True
        
    except Exception as e:
        print(f"\n❌ Filter test error: {e}")
        return False


def test_cots_query_parameter(token, query_name, fields):
    """Test COTS with query parameter"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    fields_str = "\n                ".join(fields[:5])
    
    # Test with query parameter
    query_with_param = f"""
    query {query_name}($query: COTSQuery, $pageSize: Int) {{
        {query_name}(query: $query, pageSize: $pageSize) {{
            records {{
                {fields_str}
            }}
            totalRecords
        }}
    }}
    """
    
    payload = {
        "query": query_with_param,
        "variables": {
            "query": {},
            "pageSize": 5
        }
    }
    
    try:
        res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
        res.raise_for_status()
        
        data = res.json()
        
        if 'errors' in data:
            print("❌ COTSQuery parameter not supported or incorrect type")
            return False
        else:
            print("✓ COTSQuery parameter is supported!")
            return True
        
    except Exception as e:
        print(f"❌ Query test error: {e}")
        return False


def get_all_cots_paginated(token, query_name, fields, page_size=100):
    """Fetch all COTS items with pagination"""
    all_cots = []
    page_start = 0

    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    fields_str = "\n                ".join(fields)
    
    query = f"""
    query {query_name}($pageSize: Int, $pageStart: Int) {{
        {query_name}(pageSize: $pageSize, pageStart: $pageStart) {{
            records {{
                {fields_str}
            }}
            pageSize
            pageStart
            totalRecords
        }}
    }}
    """
    
    while True:
        payload = {
            "query": query,
            "variables": {
                "pageSize": page_size,
                "pageStart": page_start
            }
        }
        
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
            res.raise_for_status()
            
            data = res.json()
            
            if 'data' in data and query_name in data['data']:
                cots_data = data['data'][query_name]
                records = cots_data['records']
                total_records = cots_data['totalRecords']
                
                all_cots.extend(records)
                
                logging.info(f"Fetched {len(records)} records (Total: {len(all_cots)}/{total_records})")
                
                # Check if we've fetched all records
                if page_start + page_size >= total_records or not records:
                    break
                
                page_start += page_size
            else:
                break
        
        except Exception as e:
            logging.error(f"Error during pagination: {e}")
            break
    
    logging.info(f"Total COTS items fetched: {len(all_cots)}")
    return all_cots


if __name__ == "__main__":
    token = get_token()
    if token:
        print("\n📦 PROSHOP COTS MODULE - COMPLETE TESTING")
        print("="*60)
        
        # Step 1: Find the correct query name
        print("\n[STEP 1] Finding correct query name...")
        query_name = test_simple_cots_query(token)
        
        if query_name:
            print(f"\n✓ Found query name: {query_name}")
            
            # Step 2: Test all possible fields
            print(f"\n[STEP 2] Testing fields for {query_name}...")
            valid_fields = test_cots_fields(token, query_name)
            
            print("\n" + "="*60)
            print(f"ALL VALID FIELDS ({len(valid_fields)}):")
            print("="*60)
            for field in valid_fields:
                print(f"  • {field}")
            
            if valid_fields:
                # Step 3: Get sample data
                print(f"\n[STEP 3] Retrieving sample {query_name}...")
                get_cots_with_valid_fields(token, query_name, valid_fields, page_size=10)
                
                # Step 4: Test filters
                print(f"\n[STEP 4] Testing filter support...")
                test_cots_with_filters(token, query_name, valid_fields)
                
                # Step 5: Test query parameter
                print(f"\n[STEP 5] Testing query parameter support...")
                test_cots_query_parameter(token, query_name, valid_fields)
                
                # Step 6: Option to fetch all
                print("\n" + "="*60)
                print("READY TO FETCH ALL COTS ITEMS")
                print("="*60)
                print("\nTo fetch all COTS items, uncomment the following:")
                print(f"# all_cots = get_all_cots_paginated(token, '{query_name}', valid_fields, page_size=100)")
                
                # Uncomment to fetch all:
                # all_cots = get_all_cots_paginated(token, query_name, valid_fields, page_size=100)
                # print(f"\n✓ Fetched all {len(all_cots)} COTS items!")
                # with open('all_cots.json', 'w') as f:
                #     json.dump(all_cots, f, indent=2)
                # print("✓ All COTS data saved to 'all_cots.json'")
                
        else:
            print("\n❌ Could not find COTS query.")
            print("\nPossible reasons:")
            print("  • COTS might be under a different module name")
            print("  • COTS items might be part of the 'parts' module with a filter")
            print("  • Your account might not have access to COTS data")
            print("\nTry checking if COTS is a type/category within the Parts module:")
            print("  • parts(filter: {partType: 'COTS'})")
            print("  • parts(filter: {type: 'Purchased'})")

In [ ]:
def test_equipment_fields_comprehensive(token):
    """Test all equipment fields to find potential work order connections"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    test_fields = [
        # Basic identification
        "number",
        "equipmentNumber",
        "internalPartNumber",
        "name",
        "equipmentName",
        "description",
        
        # Status and type
        "status",
        "statusLabel",
        "type",
        "category",
        
        # Work order related fields (these might exist!)
        "currentWorkOrder",
        "workOrderNumber",
        "assignedWorkOrder",
        "activeWorkOrder",
        "workOrders",  # Array of work orders?
        "jobNumber",
        "currentJob",
        
        # Usage and tracking
        "hoursUsed",
        "totalHours",
        "currentOperation",
        "lastOperation",
        "utilizationRate",
        "availability",
        
        # Assignment
        "assignedTo",
        "operator",
        "workCenter",
        "department",
        "location",
        
        # Technical specs
        "manufacturer",
        "model",
        "serialNumber",
        
        # Dates
        "lastUsedDate",
        "installDate",
        "purchaseDate",
        
        # Other
        "notes",
        "specifications",
        "cost",
        "value"
    ]
    
    valid_fields = []
    
    print("\n" + "="*60)
    print("TESTING EQUIPMENT FIELDS (Including Work Order Related)")
    print("="*60)
    
    for field in test_fields:
        query = f"""
        query {{
            equipments(pageSize: 5) {{
                records {{
                    {field}
                }}
                totalRecords
            }}
        }}
        """
        
        payload = {"query": query}
        
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
            res.raise_for_status()
            data = res.json()
            
            if 'errors' in data:
                print(f"  ❌ {field}: Not available")
            elif 'data' in data:
                print(f"  ✓ {field}: VALID")
                valid_fields.append(field)
                # Show sample values from multiple records
                if data['data']['equipments']['records']:
                    samples = []
                    for record in data['data']['equipments']['records'][:3]:
                        val = record.get(field)
                        if val:
                            samples.append(str(val)[:30])
                    if samples:
                        print(f"      Samples: {', '.join(samples)}")
        
        except Exception as e:
            print(f"  ❌ {field}: Error")
    
    return valid_fields


def test_nested_work_order_in_equipment(token):
    """Test if equipment has nested work order objects"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    print("\n" + "="*60)
    print("TESTING NESTED WORK ORDER OBJECTS IN EQUIPMENT")
    print("="*60)
    
    nested_tests = [
        {
            "name": "workOrders array",
            "query": """
            query {
                equipments(pageSize: 1) {
                    records {
                        number
                        workOrders {
                            workOrderNumber
                            status
                        }
                    }
                }
            }
            """
        },
        {
            "name": "currentWorkOrder object",
            "query": """
            query {
                equipments(pageSize: 1) {
                    records {
                        number
                        currentWorkOrder {
                            workOrderNumber
                            status
                        }
                    }
                }
            }
            """
        },
        {
            "name": "assignedWorkOrder object",
            "query": """
            query {
                equipments(pageSize: 1) {
                    records {
                        number
                        assignedWorkOrder {
                            workOrderNumber
                            status
                        }
                    }
                }
            }
            """
        }
    ]
    
    for test in nested_tests:
        payload = {"query": test["query"]}
        
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
            res.raise_for_status()
            data = res.json()
            
            if 'errors' in data:
                print(f"  ❌ {test['name']}: Not available")
            elif 'data' in data:
                print(f"  ✓ {test['name']}: VALID")
                print(f"      Data: {json.dumps(data['data']['equipments']['records'], indent=2)}")
                return True
        except:
            print(f"  ❌ {test['name']}: Error")
    
    return False


def get_equipment_sample_data(token, valid_fields):
    """Get comprehensive sample of equipment data"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    fields_str = "\n                ".join(valid_fields)
    
    query = f"""
    query {{
        equipments(pageSize: 20, pageStart: 0) {{
            records {{
                {fields_str}
            }}
            totalRecords
        }}
    }}
    """
    
    payload = {"query": query}
    
    try:
        res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
        res.raise_for_status()
        data = res.json()
        
        if 'data' in data:
            print("\n" + "="*60)
            print("EQUIPMENT SAMPLE DATA (20 records)")
            print("="*60)
            
            records = data['data']['equipments']['records']
            
            # Save to file for analysis
            with open('equipment_sample_20.json', 'w') as f:
                json.dump(records, f, indent=2)
            
            print(f"✓ Saved 20 equipment records to 'equipment_sample_20.json'")
            print(f"✓ Total equipment in system: {data['data']['equipments']['totalRecords']}")
            
            # Show first 3 for quick view
            for i, record in enumerate(records[:3], 1):
                print(f"\nEquipment {i}:")
                print(json.dumps(record, indent=2))
            
            return records
    
    except Exception as e:
        print(f"Error: {e}")
        return None


def analyze_equipment_work_order_relationship(token):
    """Analyze if there's a way to link equipment to work orders"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    print("\n" + "="*60)
    print("ANALYZING EQUIPMENT-WORK ORDER RELATIONSHIP")
    print("="*60)
    
    # Get one equipment record and one work order record to compare fields
    equipment_query = """
    query {
        equipments(pageSize: 1) {
            records {
                number
                name
                internalPartNumber
            }
        }
    }
    """
    
    wo_query = """
    query {
        workOrders(pageSize: 1) {
            records {
                workOrderNumber
                partPlainText
            }
        }
    }
    """
    
    print("\n🔍 Looking for linking fields...")
    print("\nStrategy 1: Check if equipment 'internalPartNumber' matches work order 'partPlainText'")
    print("Strategy 2: Check if equipment tracking happens through a separate 'usage' or 'history' module")
    print("Strategy 3: Equipment might be tracked at operation level (not available in this API)")
    
    try:
        # Get equipment sample
        eq_res = requests.post(url, headers=headers, json={"query": equipment_query}, timeout=(10, 30))
        eq_data = eq_res.json()
        
        # Get work order sample
        wo_res = requests.post(url, headers=headers, json={"query": wo_query}, timeout=(10, 30))
        wo_data = wo_res.json()
        
        print("\n📋 Sample Equipment:")
        print(json.dumps(eq_data.get('data', {}).get('equipments', {}).get('records', []), indent=2))
        
        print("\n📋 Sample Work Order:")
        print(json.dumps(wo_data.get('data', {}).get('workOrders', {}).get('records', []), indent=2))
        
    except Exception as e:
        print(f"Error: {e}")


def suggest_linking_strategies(valid_fields):
    """Suggest ways to link equipment to work orders based on available fields"""
    
    print("\n" + "="*60)
    print("SUGGESTED LINKING STRATEGIES")
    print("="*60)
    
    strategies = []
    
    if 'internalPartNumber' in valid_fields:
        strategies.append({
            "method": "Part Number Matching",
            "description": "Link equipment to work orders via partPlainText or part numbers",
            "fields": ["internalPartNumber"],
            "confidence": "Medium"
        })
    
    if 'number' in valid_fields or 'equipmentNumber' in valid_fields:
        strategies.append({
            "method": "Manual Tracking",
            "description": "Track equipment usage manually in work order notes or custom fields",
            "fields": ["number", "equipmentNumber"],
            "confidence": "Low"
        })
    
    if 'workCenter' in valid_fields or 'department' in valid_fields or 'location' in valid_fields:
        strategies.append({
            "method": "Location-Based Linking",
            "description": "Link equipment by work center, department, or location",
            "fields": ["workCenter", "department", "location"],
            "confidence": "Low"
        })
    
    if not strategies:
        strategies.append({
            "method": "Separate Tracking",
            "description": "Equipment and work orders are tracked independently in Proshop",
            "fields": [],
            "confidence": "High"
        })
    
    for i, strategy in enumerate(strategies, 1):
        print(f"\n{i}. {strategy['method']} (Confidence: {strategy['confidence']})")
        print(f"   {strategy['description']}")
        if strategy['fields']:
            print(f"   Using fields: {', '.join(strategy['fields'])}")


if __name__ == "__main__":
    token = get_token()
    if token:
        print("\n🔗 ANALYZING EQUIPMENT-WORK ORDER RELATIONSHIP")
        print("="*60)
        
        # Step 1: Test all equipment fields
        print("\n[STEP 1] Testing all equipment fields...")
        valid_fields = test_equipment_fields_comprehensive(token)
        
        print(f"\n✓ Found {len(valid_fields)} valid equipment fields")
        
        # Step 2: Test for nested work order objects
        print("\n[STEP 2] Testing for nested work order relationships...")
        has_nested_wo = test_nested_work_order_in_equipment(token)
        
        # Step 3: Get sample data
        print("\n[STEP 3] Getting equipment sample data...")
        sample_data = get_equipment_sample_data(token, valid_fields)
        
        # Step 4: Analyze relationship
        print("\n[STEP 4] Analyzing potential relationships...")
        analyze_equipment_work_order_relationship(token)
        
        # Step 5: Suggest linking strategies
        print("\n[STEP 5] Suggesting linking strategies...")
        suggest_linking_strategies(valid_fields)
        
        print("\n" + "="*60)
        print("FINAL SUMMARY")
        print("="*60)
        print(f"\n✓ Equipment Module: 727 records available")
        print(f"✓ Valid Fields: {len(valid_fields)}")
        print(f"✓ Direct WO Link: {'Yes' if has_nested_wo else 'No'}")
        
        if not has_nested_wo:
            print("\n  Equipment and Work Orders appear to be separate entities")
            print("   You may need to:")
            print("   1. Query them separately and join in your application")
            print("   2. Use part numbers or other common fields to establish relationships")
            print("   3. Check if Proshop has a separate 'Equipment Usage' or 'Time Tracking' module")

2026-01-13 13:29:23,481 - INFO - Deriving session token
2026-01-13 13:29:26,265 - INFO - succesfully derived session token



🔗 ANALYZING EQUIPMENT-WORK ORDER RELATIONSHIP

[STEP 1] Testing all equipment fields...

TESTING EQUIPMENT FIELDS (Including Work Order Related)
  ❌ number: Not available
  ✓ equipmentNumber: VALID
      Samples: 250100, 2572345, 21884
  ❌ internalPartNumber: Not available
  ❌ name: Not available
  ❌ equipmentName: Not available
  ✓ description: VALID
      Samples: DAP® Joint Sealant: 2.8 oz Tub
  ✓ status: VALID
      Samples: Active, Active, Active
  ❌ statusLabel: Not available
  ✓ type: VALID
      Samples: Thread Sealer, DAP® Joint Sealant: 2.8 oz Tub
  ❌ category: Not available
  ❌ currentWorkOrder: Not available
  ❌ workOrderNumber: Not available
  ❌ assignedWorkOrder: Not available
  ❌ activeWorkOrder: Not available
  ❌ workOrders: Not available
  ❌ jobNumber: Not available
  ❌ currentJob: Not available
  ❌ hoursUsed: Not available
  ❌ totalHours: Not available
  ❌ currentOperation: Not available
  ❌ lastOperation: Not available
  ❌ utilizationRate: Not available
  ❌ availabi

In [ ]:
def test_client_part_number_in_work_orders(token):
    """Test if clientPartNumber field exists in workOrders query"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    # Test direct clientPartNumber field
    query_direct = """
    query workOrders($pageSize: Int) {
        workOrders(pageSize: $pageSize) {
            records {
                workOrderNumber
                clientPartNumber
            }
            totalRecords
        }
    }
    """
    
    payload = {
        "query": query_direct,
        "variables": {
            "pageSize": 5
        }
    }
    
    print("\n" + "="*60)
    print("TEST 1: Direct 'clientPartNumber' field")
    print("="*60)
    
    try:
        res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
        res.raise_for_status()
        data = res.json()
        
        if 'errors' in data:
            print(f"❌ clientPartNumber: Not available")
            print(f"   Error: {data['errors'][0]['message']}")
            return False
        elif 'data' in data:
            print(f"✓ clientPartNumber: VALID")
            # Show sample values
            for i, record in enumerate(data['data']['workOrders']['records'], 1):
                client_part = record.get('clientPartNumber', 'N/A')
                wo_number = record.get('workOrderNumber', 'N/A')
                print(f"   Sample {i}: WO {wo_number} -> Client Part: {client_part}")
            return True
    
    except Exception as e:
        print(f"❌ Error testing clientPartNumber: {e}")
        return False


def test_part_related_fields(token):
    """Test all part-related fields to find alternatives"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    # All possible part-related field names
    part_fields = [
        "clientPartNumber",
        "partNumber",
        "partPlainText",
        "partRev",
        "partRevision",
        "partName",
        "partDescription",
        "customerPartNumber",
        "customerPart",
        "part",
        "partId",
        "internalPartNumber"
    ]
    
    print("\n" + "="*60)
    print("TEST 2: All Part-Related Fields")
    print("="*60)
    
    valid_part_fields = []
    
    for field in part_fields:
        query = f"""
        query workOrders($pageSize: Int) {{
            workOrders(pageSize: $pageSize) {{
                records {{
                    workOrderNumber
                    {field}
                }}
                totalRecords
            }}
        }}
        """
        
        payload = {
            "query": query,
            "variables": {
                "pageSize": 3
            }
        }
        
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
            res.raise_for_status()
            data = res.json()
            
            if 'errors' in data:
                print(f"  ❌ {field}: Not available")
            elif 'data' in data:
                print(f"  ✓ {field}: VALID")
                valid_part_fields.append(field)
                
                # Show sample values
                samples = []
                for record in data['data']['workOrders']['records']:
                    val = record.get(field)
                    if val:
                        samples.append(str(val)[:40])
                
                if samples:
                    print(f"      Samples: {', '.join(samples)}")
        
        except Exception as e:
            print(f"  ❌ {field}: Error")
    
    return valid_part_fields


def test_nested_part_object(token):
    """Test if part information is in a nested object"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    print("\n" + "="*60)
    print("TEST 3: Nested Part Objects")
    print("="*60)
    
    nested_tests = [
        {
            "name": "part { clientPartNumber }",
            "query": """
            query workOrders($pageSize: Int) {
                workOrders(pageSize: $pageSize) {
                    records {
                        workOrderNumber
                        part {
                            clientPartNumber
                            partNumber
                            name
                        }
                    }
                    totalRecords
                }
            }
            """
        },
        {
            "name": "partInfo { clientPartNumber }",
            "query": """
            query workOrders($pageSize: Int) {
                workOrders(pageSize: $pageSize) {
                    records {
                        workOrderNumber
                        partInfo {
                            clientPartNumber
                            partNumber
                        }
                    }
                    totalRecords
                }
            }
            """
        },
        {
            "name": "partDetails { clientPartNumber }",
            "query": """
            query workOrders($pageSize: Int) {
                workOrders(pageSize: $pageSize) {
                    records {
                        workOrderNumber
                        partDetails {
                            clientPartNumber
                        }
                    }
                    totalRecords
                }
            }
            """
        }
    ]
    
    valid_nested = []
    
    for test in nested_tests:
        payload = {
            "query": test["query"],
            "variables": {
                "pageSize": 2
            }
        }
        
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
            res.raise_for_status()
            data = res.json()
            
            if 'errors' in data:
                print(f"  ❌ {test['name']}: Not available")
            elif 'data' in data:
                print(f"  ✓ {test['name']}: VALID")
                valid_nested.append(test['name'])
                
                # Show sample
                for record in data['data']['workOrders']['records']:
                    print(f"      Sample: {json.dumps(record, indent=8)}")
        
        except Exception as e:
            print(f"  ❌ {test['name']}: Error")
    
    return valid_nested


def compare_with_current_query(token):
    """Compare current query with clientPartNumber added"""
    url = f"{start_url}/graphql?token={token}"
    headers = {
        "content-type": "application/json"
    }
    
    print("\n" + "="*60)
    print("TEST 4: Current Query + clientPartNumber")
    print("="*60)
    
    query_with_client_part = """
    query workOrders($filter: WorkOrderFilter, $pageSize: Int, $pageStart: Int, $query: WorkOrderQuery) {
        workOrders(filter: $filter, pageSize: $pageSize, pageStart: $pageStart, query: $query) {
            records {
                workOrderNumber
                partPlainText
                partRev
                clientPartNumber
                scheduledStartDate
                status
            }
            totalRecords
        }
    }
    """
    
    payload = {
        "query": query_with_client_part,
        "variables": {
            "filter": {"activeInventoryFlag": True},
            "pageSize": 5,
            "pageStart": 0,
            "query": {
                "lastModifiedTime": {
                    "greaterThanOrEqual": "2024-01-01T00:00:00Z"
                }
            }
        }
    }
    
    try:
        res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
        res.raise_for_status()
        data = res.json()
        
        if 'errors' in data:
            print(f"❌ Query with clientPartNumber failed")
            print(f"   Error: {data['errors']}")
            return False
        elif 'data' in data:
            print(f"✓ Query with clientPartNumber works!")
            print(f"   Total records: {data['data']['workOrders']['totalRecords']}")
            
            print("\n   Sample data:")
            for i, record in enumerate(data['data']['workOrders']['records'], 1):
                print(f"\n   Work Order {i}:")
                print(f"     WO Number: {record.get('workOrderNumber')}")
                print(f"     Part Plain Text: {record.get('partPlainText')}")
                print(f"     Client Part Number: {record.get('clientPartNumber', 'N/A')}")
                print(f"     Part Rev: {record.get('partRev')}")
            
            return True
    
    except Exception as e:
        print(f"❌ Error: {e}")
        return False


def generate_updated_query(valid_part_fields):
    """Generate the updated work order query with all valid part fields"""
    
    print("\n" + "="*60)
    print("GENERATING UPDATED QUERY")
    print("="*60)
    
    base_fields = """workOrderNumber
            scheduledStartDate
            scheduledEndDate
            dueDate
            quantityOrdered
            qtyComplete
            qtyInWIP
            qtyShipped
            qtyNotYetShipped
            dateShipped
            daysToShip
            status
            hoursCurrentTarget
            hoursTotalSpent
            setupTimeHoursActualLabel
            setupTimeHoursPlannedTarget
            setupTimeHoursPlannedVarianceLabor
            runningTimeHoursActualLabor
            runningTimeHoursPlannedTargetLabor
            runningTimeHoursPlannedVarianceLabor
            laborWIP
            standardizedLaborClass
            standardizedLaborRate
            standardProfitmMargin
            actualDollarAmount
            partPlainText
            partRev
            pfiPrice
            totalCappedWIP
            totalUncappedWIP
            type
            wipCogsLabor
            wipCogsMaterials
            wipDirectOverhead
            wipIndirectOverhead"""
    
    # Add new valid part fields
    new_fields = "\n            ".join([f for f in valid_part_fields if f not in ["partPlainText", "partRev"]])
    
    updated_query = f"""
    query workOrders($filter: WorkOrderFilter, $pageSize: Int, $pageStart: Int, $query: WorkOrderQuery) {{
        workOrders(filter: $filter, pageSize: $pageSize, pageStart: $pageStart, query: $query) {{
            records {{
                {base_fields}
                {new_fields}
            }}
            pageSize
            pageStart
            totalRecords
        }}
    }}
    """
    
    # Save to file
    with open('updated_work_order_query.txt', 'w') as f:
        f.write(updated_query)
    
    print("✓ Updated query saved to 'updated_work_order_query.txt'")
    print(f"\n✓ Added {len([f for f in valid_part_fields if f not in ['partPlainText', 'partRev']])} new part fields:")
    for field in valid_part_fields:
        if field not in ["partPlainText", "partRev"]:
            print(f"   • {field}")
    
    return updated_query


if __name__ == "__main__":
    token = get_token()
    if token:
        print("\n🔍 TESTING clientPartNumber IN WORK ORDERS")
        print("="*60)
        
        # Test 1: Direct field test
        has_client_part = test_client_part_number_in_work_orders(token)
        
        # Test 2: All part-related fields
        valid_part_fields = test_part_related_fields(token)
        
        # Test 3: Nested objects
        nested_fields = test_nested_part_object(token)
        
        # Test 4: Test with your current query structure
        works_with_filters = compare_with_current_query(token)
        
        # Summary
        print("\n" + "="*60)
        print("SUMMARY")
        print("="*60)
        
        if has_client_part:
            print("\n✅ clientPartNumber EXISTS in work orders!")
            print("   You can add it to your query.")
        else:
            print("\n❌ clientPartNumber does NOT exist in work orders")
        
        if valid_part_fields:
            print(f"\n✓ Found {len(valid_part_fields)} valid part-related fields:")
            for field in valid_part_fields:
                print(f"   • {field}")
            
            # Generate updated query
            generate_updated_query(valid_part_fields)
        
        if nested_fields:
            print(f"\n✓ Found nested part objects:")
            for field in nested_fields:
                print(f"   • {field}")
        
        print("\n" + "="*60)
        print("RECOMMENDATION")
        print("="*60)
        
        if 'clientPartNumber' in valid_part_fields:
            print("\n Add 'clientPartNumber' to your work order query")
            print("   It's available as a direct field!")
        elif valid_part_fields:
            print(f"\n  'clientPartNumber' not found, but you have these alternatives:")
            for field in valid_part_fields:
                print(f"   • {field}")
        else:
            print("\n No additional part fields found beyond partPlainText and partRev")


🔍 TESTING clientPartNumber IN WORK ORDERS

TEST 1: Direct 'clientPartNumber' field
❌ clientPartNumber: Not available
   Error: Unexpected field clientPartNumber in selection list for records

TEST 2: All Part-Related Fields
  ❌ clientPartNumber: Not available
  ❌ partNumber: Not available
  ✓ partPlainText: VALID
      Samples: CLIENT_VENDOR1-TIP-0000, TEX1-39234-43021-1, TEX1-39234-43021-1
  ✓ partRev: VALID
      Samples: A, B, B
  ❌ partRevision: Not available
  ❌ partName: Not available
  ❌ partDescription: Not available
  ❌ customerPartNumber: Not available
  ❌ customerPart: Not available
  ❌ part: Not available
  ❌ partId: Not available
  ❌ internalPartNumber: Not available

TEST 3: Nested Part Objects
  ❌ part { clientPartNumber }: Not available
  ❌ partInfo { clientPartNumber }: Not available
  ❌ partDetails { clientPartNumber }: Not available

TEST 4: Current Query + clientPartNumber
❌ Query with clientPartNumber failed
   Error: [{'message': 'Unexpected field clientPartNumbe

In [ ]:



def test_internal_part_number_in_invoices(token):
    """
    Test whether internalPartNumber (or equivalent linking fields)
    exist directly on the Invoice API.
    """
    url = f"{start_url}/graphql?token={token}"
    headers = {"content-type": "application/json"}

    query = """
    query invoices($pageSize: Int) {
        invoices(pageSize: $pageSize) {
            records {
                invoiceId

                # --- Direct part references (if exposed) ---
                internalPartNumber
                partNumber
                clientPartNumber
                partRev
                revision

                # --- Manufacturing linkage fields ---
                workOrderId
                jobId
                salesOrderId
            }
            totalRecords
        }
    }
    """

    payload = {
        "query": query,
        "variables": {"pageSize": 5}
    }

    print("\n" + "=" * 70)
    print("TEST 1: internalPartNumber & Manufacturing Link Fields on Invoices")
    print("=" * 70)

    try:
        res = requests.post(url, headers=headers, json=payload, timeout=(10, 30))
        res.raise_for_status()
        data = res.json()

        if "errors" in data:
            print("❌ Invoice API does not expose one or more requested fields")
            for err in data["errors"]:
                print(f"   Error: {err['message']}")
            return False

        records = data["data"]["invoices"]["records"]
        if not records:
            print("❌ No invoice records returned")
            return False

        print("✓ Invoice query executed successfully\n")

        for i, r in enumerate(records, 1):
            print(f"Invoice Sample {i}")
            print(f"  Invoice ID           : {r.get('invoiceId')}")
            print(f"  internalPartNumber   : {r.get('internalPartNumber')}")
            print(f"  partNumber           : {r.get('partNumber')}")
            print(f"  clientPartNumber     : {r.get('clientPartNumber')}")
            print(f"  partRev / revision   : {r.get('partRev') or r.get('revision')}")
            print(f"  workOrderId          : {r.get('workOrderId')}")
            print(f"  jobId                : {r.get('jobId')}")
            print(f"  salesOrderId         : {r.get('salesOrderId')}")
            print("-" * 50)

        return True

    except Exception as e:
        print(f"❌ Error testing invoice linkage fields: {e}")
        return False


def test_invoice_to_workorder_link(token):
    """
    Validate whether invoices can be joined to work orders,
    which is the canonical path to Part / PartRevision in ProShop.
    """
    url = f"{start_url}/graphql?token={token}"
    headers = {"content-type": "application/json"}

    query = """
    query invoices($pageSize: Int) {
        invoices(pageSize: $pageSize) {
            records {
                invoiceId
                workOrderId
            }
        }
    }
    """

    print("\n" + "=" * 70)
    print("TEST 2: Invoice → WorkOrder Link Validation")
    print("=" * 70)

    try:
        res = requests.post(
            url,
            headers=headers,
            json={"query": query, "variables": {"pageSize": 5}},
            timeout=(10, 30)
        )
        res.raise_for_status()
        data = res.json()

        if "errors" in data:
            print("❌ workOrderId not available on invoices")
            return False

        records = data["data"]["invoices"]["records"]

        linked = [r for r in records if r.get("workOrderId")]

        if not linked:
            print("❌ Invoices do not link to Work Orders")
            print("   You must join via Sales Orders or Job records instead")
            return False

        print("✓ Invoices link to Work Orders")
        for r in linked:
            print(f"  Invoice {r['invoiceId']} → WorkOrder {r['workOrderId']}")

        return True

    except Exception as e:
        print(f"❌ Error validating invoice → work order link: {e}")
        return False


def recommended_proshop_join_model():
    """
    Documents the correct ProShop data model path.
    """
    print("\n" + "=" * 70)
    print("RECOMMENDED PROSHOP PART TRACEABILITY MODEL")
    print("=" * 70)

    print("""
Invoice
  └── workOrderId
        └── WorkOrder
              ├── internalPartNumber
              ├── partNumber
              ├── partRev
              └── jobId
                    └── Part / PartRevision

✔ internalPartNumber is normally mastered on:
  - Part
  - PartRevision
  - WorkOrder

✖ Invoice is a financial document and often does NOT
  carry part master data directly.

BEST PRACTICE:
1. Pull invoices
2. Join to work orders using workOrderId
3. Join work orders to part / partRevision
""")


if __name__ == "__main__":
    

    token = get_token()

    if not token:
        raise RuntimeError("Token acquisition failed")

    print("\n🔍 MANUFACTURING PROSHOP INVOICE FIELD VALIDATION")

    test_internal_part_number_in_invoices(token)
    test_invoice_to_workorder_link(token)
    recommended_proshop_join_model()



🔍 MANUFACTURING PROSHOP INVOICE FIELD VALIDATION

TEST 1: internalPartNumber & Manufacturing Link Fields on Invoices
❌ Invoice API does not expose one or more requested fields
   Error: Unexpected field internalPartNumber in selection list for records

TEST 2: Invoice → WorkOrder Link Validation
❌ workOrderId not available on invoices

RECOMMENDED PROSHOP PART TRACEABILITY MODEL

Invoice
  └── workOrderId
        └── WorkOrder
              ├── internalPartNumber
              ├── partNumber
              ├── partRev
              └── jobId
                    └── Part / PartRevision

✔ internalPartNumber is normally mastered on:
  - Part
  - PartRevision
  - WorkOrder

✖ Invoice is a financial document and often does NOT
  carry part master data directly.

BEST PRACTICE:
1. Pull invoices
2. Join to work orders using workOrderId
3. Join work orders to part / partRevision



In [ ]:
import json


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def inspect_sales_order_schema(token):
    """Introspect the GraphQL schema to find available fields in SalesOrder"""
    url = f"{start_url}/graphql?token={token}"
    header = {"content-type": "application/json"}
    
    # Introspection query to get SalesOrder information
    introspection_query = """
    {
        __type(name: "SalesOrder") {
            name
            fields(includeDeprecated: false) {
                name
                type {
                    name
                    kind
                    ofType {
                        name
                        kind
                        ofType {
                            name
                            kind
                        }
                    }
                }
            }
        }
    }
    """
    
    try:
        payload = {"query": introspection_query}
        res = requests.post(url, headers=header, json=payload, timeout=10)
        res.raise_for_status()
        
        data = res.json()
        
        if 'errors' in data:
            logging.error(f"GraphQL Errors: {data['errors']}")
            return search_types(token)
        
        if data.get('data') is None or data['data'].get('__type') is None:
            logging.warning("Type 'SalesOrder' not found. Searching for available types...")
            return search_types(token)
        
        print("\n" + "="*80)
        print("AVAILABLE FIELDS IN SalesOrder:")
        print("="*80)
        
        object_fields = []
        scalar_fields = []
        
        for field in data['data']['__type']['fields']:
            field_name = field['name']
            field_type = get_type_string(field['type'])
            
            # Categorize fields
            if field['type']['kind'] == 'OBJECT' or (field['type'].get('ofType', {}).get('kind') == 'OBJECT'):
                object_fields.append((field_name, field_type, field['type']))
            else:
                scalar_fields.append((field_name, field_type))
        
        # Print scalar fields
        print("\n📌 SCALAR FIELDS (can be queried directly):")
        print("-" * 80)
        for field_name, field_type in sorted(scalar_fields):
            print(f"  • {field_name}: {field_type}")
        
        # Print object fields
        if object_fields:
            print("\n" + "="*80)
            print("🔗 NESTED OBJECT FIELDS (need sub-selections):")
            print("="*80)
            for field_name, field_type, field_type_obj in sorted(object_fields):
                print(f"\n  → {field_name}: {field_type}")
                inspect_nested_type(token, field_name, field_type_obj)
        
        # Generate sample query
        print("\n" + "="*80)
        print("📝 SAMPLE QUERY YOU CAN USE:")
        print("="*80)
        generate_sample_query(scalar_fields, object_fields)
        
    except Exception as e:
        logging.error(f"Error: {e}")

def get_type_string(type_obj):
    """Convert GraphQL type to readable string"""
    if type_obj['kind'] == 'NON_NULL':
        return get_type_string(type_obj['ofType']) + "!"
    elif type_obj['kind'] == 'LIST':
        return "[" + get_type_string(type_obj['ofType']) + "]"
    else:
        return type_obj['name'] or 'Unknown'

def inspect_nested_type(token, field_name, field_type):
    """Introspect a nested object type"""
    # Get the actual type name
    type_name = field_type['name']
    if not type_name and field_type.get('ofType'):
        type_name = field_type['ofType']['name']
    
    if not type_name:
        return

    url = f"{start_url}/graphql?token={token}"
    header = {"content-type": "application/json"}
    
    query = f"""
    {{
        __type(name: "{type_name}") {{
            name
            fields(includeDeprecated: false) {{
                name
                type {{
                    name
                    kind
                    ofType {{
                        name
                        kind
                    }}
                }}
            }}
        }}
    }}
    """
    
    try:
        payload = {"query": query}
        res = requests.post(url, headers=header, json=payload, timeout=10)
        data = res.json()
        
        if data.get('data', {}).get('__type'):
            print(f"    ├─ Fields available in {type_name}:")
            for field in data['data']['__type']['fields']:
                print(f"    │  • {field['name']}: {get_type_string(field['type'])}")
    except Exception as e:
        logging.debug(f"Could not inspect {type_name}: {e}")

def search_types(token):
    """Search for available types in the schema"""
    url = f"{start_url}/graphql?token={token}"
    header = {"content-type": "application/json"}
    
    query = """
    {
        __schema {
            types {
                name
                kind
            }
        }
    }
    """
    
    try:
        payload = {"query": query}
        res = requests.post(url, headers=header, json=payload, timeout=10)
        data = res.json()
        
        print("\n" + "="*80)
        print("AVAILABLE TYPES IN SCHEMA:")
        print("="*80)
        
        # Filter and sort
        user_types = [t for t in data['data']['__schema']['types'] if not t['name'].startswith('__')]
        user_types.sort(key=lambda x: x['name'])
        
        for type_obj in user_types:
            print(f"  • {type_obj['name']} ({type_obj['kind']})")
            
        # Look for order-related types
        print("\n" + "="*80)
        print("💡 ORDER-RELATED TYPES FOUND:")
        print("="*80)
        order_types = [t['name'] for t in user_types if 'order' in t['name'].lower()]
        for type_name in order_types:
            print(f"  ✓ {type_name}")
            
    except Exception as e:
        logging.error(f"Error: {e}")

def generate_sample_query(scalar_fields, object_fields):
    """Generate a sample GraphQL query"""
    print("\n```graphql")
    print("query salesOrders($pageSize: Int, $pageStart: Int) {")
    print("  salesOrders(pageSize: $pageSize, pageStart: $pageStart) {")
    print("    records {")
    
    # Add scalar fields
    for field_name, _ in scalar_fields[:10]:  # First 10 scalar fields
        print(f"      {field_name}")
    
    if object_fields:
        print("\n      # Nested objects - uncomment what you need:")
        for field_name, _, _ in object_fields[:5]:
            print(f"      # {field_name} {{")
            print(f"      #   # Add fields here")
            print(f"      # }}")
    
    print("    }")
    print("    pageSize")
    print("    pageStart")
    print("    totalRecords")
    print("  }")
    print("}")
    print("```\n")

if __name__ == "__main__":
    token = get_token()
    if token:
        logging.info("Inspecting SalesOrder schema...")
        inspect_sales_order_schema(token)
    else:
        logging.error("Could not get token")


AVAILABLE TYPES IN SCHEMA:
  • AddBillInput (INPUT_OBJECT)
  • AddBillItemsDataInput (INPUT_OBJECT)
  • AddCOTSApprovedBrandsCostsTableInput (INPUT_OBJECT)
  • AddCOTSApprovedBrandsInput (INPUT_OBJECT)
  • AddCOTSInput (INPUT_OBJECT)
  • AddClassificationInput (INPUT_OBJECT)
  • AddCompanyPositionInput (INPUT_OBJECT)
  • AddCompanyPositionRequirementsDataInput (INPUT_OBJECT)
  • AddContactInput (INPUT_OBJECT)
  • AddCorrectiveActionRequestInput (INPUT_OBJECT)
  • AddCustomerPoInput (INPUT_OBJECT)
  • AddDocumentInput (INPUT_OBJECT)
  • AddEquipmentInput (INPUT_OBJECT)
  • AddEstimateInput (INPUT_OBJECT)
  • AddInvoiceInput (INPUT_OBJECT)
  • AddNCRInput (INPUT_OBJECT)
  • AddPackingSlipInput (INPUT_OBJECT)
  • AddPartInput (INPUT_OBJECT)
  • AddPreventiveActionRequestInput (INPUT_OBJECT)
  • AddPurchaseOrderInput (INPUT_OBJECT)
  • AddQualityManualInput (INPUT_OBJECT)
  • AddQualityProcedureInput (INPUT_OBJECT)
  • AddQuoteInput (INPUT_OBJECT)
  • AddRMAInput (INPUT_OBJECT)
  • AddRTA

In [ ]:
def inspect_type(token, type_name):
    """Introspect any GraphQL type"""
    url = f"{start_url}/graphql?token={token}"
    header = {"content-type": "application/json"}
    
    introspection_query = f"""
    {{
        __type(name: "{type_name}") {{
            name
            fields(includeDeprecated: false) {{
                name
                type {{
                    name
                    kind
                    ofType {{
                        name
                        kind
                        ofType {{
                            name
                            kind
                        }}
                    }}
                }}
            }}
        }}
    }}
    """
    
    try:
        payload = {"query": introspection_query}
        res = requests.post(url, headers=header, json=payload, timeout=10)
        res.raise_for_status()
        
        data = res.json()
        
        if 'errors' in data:
            logging.error(f"GraphQL Errors for {type_name}: {data['errors']}")
            return None
        
        if data.get('data', {}).get('__type') is None:
            logging.warning(f"Type '{type_name}' not found")
            return None
        
        return data['data']['__type']
        
    except Exception as e:
        logging.error(f"Error inspecting {type_name}: {e}")
        return None

def get_type_string(type_obj):
    """Convert GraphQL type to readable string"""
    if type_obj['kind'] == 'NON_NULL':
        return get_type_string(type_obj['ofType']) + "!"
    elif type_obj['kind'] == 'LIST':
        return "[" + get_type_string(type_obj['ofType']) + "]"
    else:
        return type_obj['name'] or 'Unknown'

def print_type_fields(type_data, highlight_keywords=None):
    """Pretty print type fields with optional highlighting"""
    if not type_data:
        return
    
    if highlight_keywords is None:
        highlight_keywords = ['po', 'order', 'customer', 'invoice', 'reference', 'number', 'id', 'link']
    
    print(f"\n{'='*90}")
    print(f"FIELDS IN {type_data['name'].upper()}")
    print(f"{'='*90}")
    
    # Separate fields
    important_fields = []
    other_fields = []
    
    for field in type_data['fields']:
        field_name = field['name'].lower()
        field_type = get_type_string(field['type'])
        
        # Check if field matches keywords
        is_important = any(kw in field_name for kw in highlight_keywords)
        
        if is_important:
            important_fields.append((field['name'], field_type))
        else:
            other_fields.append((field['name'], field_type))
    
    # Print important fields first
    if important_fields:
        print("\n🔴 POTENTIALLY LINKING FIELDS (may connect to other modules):")
        print("-" * 90)
        for fname, ftype in sorted(important_fields):
            print(f"  ⭐ {fname}: {ftype}")
    
    # Print all fields
    print("\n📋 ALL FIELDS:")
    print("-" * 90)
    for fname, ftype in sorted(other_fields):
        print(f"  • {fname}: {ftype}")

def find_common_fields(type1_data, type2_data):
    """Find common fields between two types"""
    if not type1_data or not type2_data:
        return
    
    fields1 = {f['name'] for f in type1_data['fields']}
    fields2 = {f['name'] for f in type2_data['fields']}
    
    common = fields1.intersection(fields2)
    
    if common:
        print(f"\n{'='*90}")
        print(f"⚡ COMMON FIELDS BETWEEN {type1_data['name']} & {type2_data['name']}")
        print(f"{'='*90}")
        for field in sorted(common):
            print(f"  🔗 {field}")

if __name__ == "__main__":
    token = get_token()
    if not token:
        logging.error("Could not get token")
        exit(1)
    
    logging.info("Inspecting WorkOrder, Invoice, and PurchaseOrder types...")
    
    # Inspect WorkOrder
    print("\n" + "="*90)
    print("STEP 1: Analyzing WorkOrder")
    print("="*90)
    wo_data = inspect_type(token, "WorkOrder")
    print_type_fields(wo_data)
    
    # Inspect Invoice
    print("\n" + "="*90)
    print("STEP 2: Analyzing Invoice")
    print("="*90)
    inv_data = inspect_type(token, "Invoice")
    print_type_fields(inv_data)
    
    # Inspect PurchaseOrder
    print("\n" + "="*90)
    print("STEP 3: Analyzing PurchaseOrder")
    print("="*90)
    po_data = inspect_type(token, "PurchaseOrder")
    print_type_fields(po_data)
    
    # Find common fields
    print("\n" + "="*90)
    print("STEP 4: Finding Connection Points")
    print("="*90)
    find_common_fields(wo_data, inv_data)
    find_common_fields(wo_data, po_data)
    find_common_fields(inv_data, po_data)
    
    # Summary
    print("\n" + "="*90)
    print("📊 SUMMARY & RECOMMENDATIONS")
    print("="*90)
    print("""
Look for these patterns in the output above:

1. DIRECT LINKS (Best case):
   ⭐ Fields named: clientPONumber, customerPONumber, invoiceNumber, 
                   workOrderNumber, purchaseOrderNumber, etc.

2. PART-BASED LINKS (Alternative):
   ⭐ If WorkOrder has 'part' or 'billOfMaterials'
   ⭐ And Invoice has 'part' 
   ⭐ And CustomerPO has 'partsOrdered'
   → You can link via Part ID

3. COST LINKING (Financial):
   ⭐ WorkOrderCostDisbursement (Manufacturing cost)
   ⭐ Invoice.totalAmount (Sale price)
   ⭐ PurchaseOrder amounts (Material cost)

4. TEMPORAL LINKING (Fallback):
   ⭐ Use dates: createdDate, invoiceDate, completionDate
   ⭐ Combined with Part ID for fuzzy matching
    """)


STEP 1: Analyzing WorkOrder

FIELDS IN WORKORDER

🔴 POTENTIALLY LINKING FIELDS (may connect to other modules):
------------------------------------------------------------------------------------------
  ⭐ asAcceptanceReportNumber1: String
  ⭐ asAcceptanceReportNumber2: String
  ⭐ asAcceptanceReportNumber3: String
  ⭐ asBaselinePartNumber: String
  ⭐ asDrawingNumber: String
  ⭐ asFunctionalTestNumber1: String
  ⭐ asFunctionalTestNumber2: String
  ⭐ asFunctionalTestNumber3: String
  ⭐ asSerialNumber: String
  ⭐ bulkImportId: String
  ⭐ certificationNumber: String
  ⭐ customer: Contact
  ⭐ customerPONumber: CustomerPO
  ⭐ customerPONumberPlainText: String
  ⭐ customerPlainText: String
  ⭐ drawingNumber: String
  ⭐ earlyAsPossible: Boolean
  ⭐ faiCustomerApproval: User
  ⭐ faiCustomerApprovalBy: User
  ⭐ faiCustomerApprovalByPlainText: String
  ⭐ faiCustomerApprovalDate: String
  ⭐ faiCustomerApprovalPlainText: String
  ⭐ legacyId: String!
  ⭐ oopOrdered: Float
  ⭐ orderStatusInfo: Pagin